# TP2 - Módulo 2
## Notebook 03 - Preprocesamiento y Feature Engineering

**Objetivo:** Limpiar los datos, crear nuevas variables derivadas, codificar variables categóricas, escalar y guardar el pipeline para reutilizarlo en predicción.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_excel('../data/raw/dataset_etiquetado.xlsx')
df_predict = pd.read_excel('../data/raw/dataset_sin_etiquetar.xlsx')

print('Labeled:', df.shape)
print('Unlabeled:', df_predict.shape)

Labeled: (50000, 27)
Unlabeled: (5692, 26)


## 1. Función de preprocesamiento reutilizable

In [3]:
def preprocess(df, is_train=True):
    """Aplica todas las transformaciones de feature engineering.
    Retorna DataFrame procesado sin el target.
    """
    df = df.copy()

    # --- Eliminar ID (no predictivo) ---
    df = df.drop(columns=['ID'], errors='ignore')

    # --- Feature Engineering ---

    # BMI
    df['BMI'] = df['weight(kg)'] / (df['height(cm)'] / 100) ** 2

    # Presión de pulso (diferencial)
    df['pulse_pressure'] = df['systolic'] - df['relaxation']

    # Razón HDL/Colesterol (indica calidad del colesterol)
    df['HDL_ratio'] = df['HDL'] / (df['Cholesterol'] + 1e-6)

    # Transformaciones log para variables con skew positivo
    for col in ['triglyceride', 'Gtp', 'ALT', 'AST', 'serum creatinine',
                'fasting blood sugar', 'LDL', 'Cholesterol']:
        df[f'log_{col}'] = np.log1p(df[col])

    # Binning de edad
    df['age_group'] = pd.cut(df['age'], bins=[0, 35, 45, 55, 65, 100],
                              labels=[0, 1, 2, 3, 4]).astype(int)

    # Interacción: gender × hemoglobin (los hombres tienen mayor hemoglobina y fuman más)
    df['gender_num'] = (df['gender'] == 'M').astype(int)
    df['gender_hemo'] = df['gender_num'] * df['hemoglobin']

    # --- Codificar categóricas ---
    df['gender'] = df['gender'].map({'M': 1, 'F': 0}).astype(int)
    df['oral']   = df['oral'].map({'Y': 1, 'N': 0}).astype(int)
    df['tartar'] = df['tartar'].map({'Y': 1, 'N': 0}).astype(int)

    return df

## 2. Aplicar preprocesamiento

In [4]:
# Guardar el target antes de procesar
y = df['smoking'].copy()

# Aplicar preprocesamiento al dataset etiquetado (sin columna smoking)
df_proc = preprocess(df.drop(columns=['smoking']))
df_predict_proc = preprocess(df_predict)

print('Features procesadas:', df_proc.shape)
print('Unlabeled procesado:', df_predict_proc.shape)

Features procesadas: (50000, 39)
Unlabeled procesado: (5692, 39)


In [5]:
df_proc.head()

,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,fasting blood sugar,Cholesterol,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,BMI,pulse_pressure,HDL_ratio,log_triglyceride,log_Gtp,log_ALT,log_AST,log_serum creatinine,log_fasting blood sugar,log_LDL,log_Cholesterol,age_group,gender_num,gender_hemo
0,0,40,155,60,3.377083,0.043056,0.041667,0.041667,0.041667,4.750000,3.041667,3.916667,8.958333,3.416667,3.041667,5.250000,0.506250,0.041667,0.004861,0.750000,0.791667,1.125000,1,0,1,24.973985,1.708333,0.339535,1.485385,0.753772,0.583146,0.559616,0.004849,1.592631,1.832581,2.298410,1,0,0.000000
1,0,40,160,60,3.375000,0.005556,0.004167,0.041667,0.041667,4.958333,2.916667,5.416667,8.000000,4.791667,1.750000,5.291667,0.504861,0.041667,0.004167,0.916667,0.791667,0.750000,1,0,1,23.437500,2.041667,0.218750,1.756420,0.559616,0.583146,0.650588,0.004158,1.858899,1.839226,2.197225,1,0,0.000000
2,1,55,170,60,3.333333,0.005556,0.005556,0.041667,0.041667,5.750000,3.583333,3.708333,10.083333,7.583333,2.291667,6.291667,0.630556,0.041667,0.041667,0.875000,0.666667,0.916667,1,0,0,20.761246,2.166667,0.227273,2.149822,0.650588,0.510826,0.628609,0.040822,1.549334,1.986732,2.405442,2,1,0.630556
3,1,40,165,70,3.666667,0.045139,0.045139,0.041667,0.041667,4.166667,2.500000,4.000000,13.416667,10.583333,1.875000,9.416667,0.588194,0.041667,0.041667,0.791667,1.083333,0.750000,1,0,1,25.711662,1.666667,0.139752,2.449567,0.559616,0.733969,0.583146,0.040822,1.609438,2.343407,2.668385,1,1,0.588194
4,0,40,155,60,3.583333,0.041667,0.041667,0.041667,0.041667,5.000000,3.083333,3.333333,7.666667,3.083333,2.583333,4.458333,0.503472,0.041667,0.004167,0.666667,0.583333,0.916667,1,0,0,24.973985,1.916667,0.336956,1.406914,0.650588,0.459532,0.510826,0.004158,1.466337,1.697143,2.159484,1,0,0.000000


In [6]:
df_proc.dtypes

gender                       int64
age                          int64
height(cm)                   int64
weight(kg)                   int64
waist(cm)                  float64
eyesight(left)             float64
eyesight(right)            float64
hearing(left)              float64
hearing(right)             float64
systolic                   float64
relaxation                 float64
fasting blood sugar        float64
Cholesterol                float64
triglyceride               float64
HDL                        float64
LDL                        float64
hemoglobin                 float64
Urine protein              float64
serum creatinine           float64
AST                        float64
ALT                        float64
Gtp                        float64
oral                         int64
dental caries                int64
tartar                       int64
BMI                        float64
pulse_pressure             float64
HDL_ratio                  float64
log_triglyceride    

## 3. Train / Test Split estratificado

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df_proc, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Distribución train:')
print(y_train.value_counts(normalize=True).round(3))
print('Distribución test:')
print(y_test.value_counts(normalize=True).round(3))

Train: (40000, 39) | Test: (10000, 39)
Distribución train:
smoking
0    0.633
1    0.367
Name: proportion, dtype: float64
Distribución test:
smoking
0    0.633
1    0.367
Name: proportion, dtype: float64


## 4. Escalado (StandardScaler)

> **Importante:** El scaler se ajusta SOLO sobre X_train y luego se aplica a X_test y a los datos de predicción.

In [8]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

df_predict_scaled = pd.DataFrame(
    scaler.transform(df_predict_proc),
    columns=df_predict_proc.columns
)

print('Escalado completado')
print('X_train_scaled:', X_train_scaled.shape)
print('df_predict_scaled:', df_predict_scaled.shape)

Escalado completado
X_train_scaled: (40000, 39)
df_predict_scaled: (5692, 39)


## 5. Guardar artefactos del pipeline

In [9]:
# Guardar datos procesados
X_train.to_parquet('../data/processed/X_train.parquet', index=True, compression='snappy')
X_test.to_parquet('../data/processed/X_test.parquet', index=True, compression='snappy')
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv', index=True)
df_predict_proc.to_parquet('../data/processed/df_predict_proc.parquet', index=False, compression='snappy')

# Guardar scaler
joblib.dump(scaler, '../models/scaler.joblib')

# Guardar la función de preprocesamiento como referencia
import inspect
with open('../models/preprocess_fn.py', 'w') as f:
    f.write(inspect.getsource(preprocess))

print('✓ Datos procesados guardados en data/processed/')
print('✓ Scaler guardado en models/scaler.joblib')

✓ Datos procesados guardados en data/processed/
✓ Scaler guardado en models/scaler.joblib


## 6. Resumen de features finales

In [10]:
print(f'Total de features: {X_train.shape[1]}')
print('Features:', X_train.columns.tolist())

Total de features: 39
Features: ['gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'oral', 'dental caries', 'tartar', 'BMI', 'pulse_pressure', 'HDL_ratio', 'log_triglyceride', 'log_Gtp', 'log_ALT', 'log_AST', 'log_serum creatinine', 'log_fasting blood sugar', 'log_LDL', 'log_Cholesterol', 'age_group', 'gender_num', 'gender_hemo']
